# Контрольне порівняння AnyVisLoc: авторський runner → адаптер → matcher → prior

Окремий notebook для пошуку причини слабкої локалізації. Лише **чисті зображення**, ті самі явно задані кадри, однакові native RGB/K/dist і фіксований commit AnyVisLoc. Основний notebook видимості не запускається.

| Етап | Вхід і змінна під перевіркою |
| --- | --- |
| A_author_Roma | Оригінальне тіло `run_avl_match_loc.process_one_sample`, авторські GT crop/resize, RoMa. Спостереження лише записує входи/виходи. |
| B_adapter_replay | Той самий GT crop і записані відповідності A через production `AnyVisLocBackend.match_candidate`. Перевірка координат, DSM та PnP. |
| B_adapter_live | Ті самі пікселі, ваги RoMa й seed; відповідності обчислюються повторно через адаптер. |
| C_SP_LG | Ті самі зображення matcher, що в A/B; замінено лише RoMa на SuperPoint/LightGlue. |
| C_SP_LG_p | Додатковий контроль: вимкнено yaw-поворот авторського crop. |
| D0_search_exact_pose | CAMP і геометрія пошуку проєкту; точна поза як діагностичний prior, радіус пошуку той самий, що в D1. |
| D1_search_noisy_pose | Той самий пошук і SP/LG, але зашумлений prior: σXY=20 м, профіль main. |

**A/B/C/D0 використовують еталон для підготовки входів. Це діагностичні контролі, не оцінка GPS-denied навігації.** У D1 еталон потрібний тільки для симуляції шуму та оцінювання після inference; правильний crop не передається.

`aerial` і `satellite` аналізуються окремо. Авторські параметри читаються з runner за зафіксованим commit; це відтворення коду, а не заява про відтворення конкретної таблиці статті чи оригінального програмного середовища. Для Python 3.13 використовується сумісне сучасніше середовище.

Артефакти: CSV, JSON, відповідності NPZ, хеші входів, авторські картинки та графіки. Випадкові значення не підставляються замість невдалих запусків. Технічна помилка зупиняє виконання; успішно завершені записи залишаються на диску.


## 1. Каталоги й фіксовані кадри

Скопіюйте всю оновлену папку `scripts/visibility_localization` у ту саму папку Google Drive, яку використовував основний notebook. Виберіть GPU runtime та почніть із нового Python-сеансу. Ваги RoMa/DINOv2 великі; завантаження й виконання потребують часу та місця. Код/датасет повторно використовуються з попереднього робочого каталогу, якщо він ще доступний.


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
WORK_ROOT = Path('/content/visibility_localization')
UPSTREAM_ROOT = WORK_ROOT/'Benchmark'
DATASET_ROOT = WORK_ROOT/'data'
ARCHIVE_DIR = WORK_ROOT/'archives'
CODE_ROOT = Path('/content/drive/MyDrive/PhD/proj/scripts')
PACKAGE_DIR = CODE_ROOT/'visibility_localization'
SAVE_RESULTS_TO_DRIVE = False
RESULTS_ROOT = (Path('/content/drive/MyDrive/PhD/visibility_results')
                if SAVE_RESULTS_TO_DRIVE else WORK_ROOT/'results')
CONTROL_RUN_TAG = 'anyvisloc-controls-v2'
RUN_TAG = CONTROL_RUN_TAG
REFERENCE_MODES = ['satellite', 'aerial']
# Ті самі три кадри з діагностики visibility-v3; перелік не змінюється за результатами.
SAMPLE_IDS = {'Scene_05': ['L05_0969'], 'Scene_08': ['L08_0049'], 'Scene_11': ['L11_0545']}
ARCHIVES = {scene: f'/content/drive/MyDrive/PhD/proj/AnyVisLoc/{scene}.zip' for scene in SAMPLE_IDS}
CAMP_LOCAL_PATH = None
for name in ['CODE_ROOT','UPSTREAM_ROOT','DATASET_ROOT','RESULTS_ROOT']:
    print(name, '->', globals()[name])


## 2. Підключення Drive та залежності

Використовується той самий вибір PyTorch для Python 3.10–3.13, що в notebook видимості, плюс залежності контрольного запуску. Після оновлення вже імпортованих бібліотек виконайте Restart session та почніть зверху. Не встановлюйте upstream `requirements.txt` поверх цього середовища: його обмеження NumPy <2 несумісне з Python 3.13.


In [ ]:
import os, sys
from google.colab import drive
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
drive.mount('/content/drive')
required = ['ablation.py','backend.py','core.py','data.py','campaign.py','colab.py',
            'analysis.py','check_upstream.py','test_experiment.py','test_ablation.py',
            'requirements-colab.txt','requirements-ablation-colab.txt']
missing = [f for f in required if not (PACKAGE_DIR/f).is_file()]
if missing:
    raise FileNotFoundError(f'Оновіть всю папку visibility_localization у Drive. Відсутні: {missing}')
sys.pycache_prefix = str(WORK_ROOT/'pycache')
os.environ['PYTHONPYCACHEPREFIX'] = sys.pycache_prefix
sys.path.insert(0, str(CODE_ROOT))
import visibility_localization
if Path(visibility_localization.__file__).resolve().parent != PACKAGE_DIR.resolve():
    raise RuntimeError('Імпортовано іншу копію пакета; перезапустіть сеанс')


In [ ]:
import sys, subprocess, importlib.metadata
from pathlib import Path
from packaging.requirements import Requirement

python_version = sys.version_info[:2]
if not (3, 10) <= python_version <= (3, 13):
    raise RuntimeError(f'Підтримується Python 3.10–3.13; поточний: {sys.version}')
if python_version == (3, 13):
    torch_version, vision_version, cuda_build = '2.6.0', '0.21.0', 'cu126'
    numpy_version = '2.1.3'
else:
    torch_version, vision_version, cuda_build = '2.5.1', '0.20.1', 'cu121'
    numpy_version = '1.26.4'
print(f'Python {sys.version.split()[0]}: torch {torch_version}, '
      f'torchvision {vision_version}, {cuda_build}, NumPy {numpy_version}', flush=True)
requirements = PACKAGE_DIR / 'requirements-colab.txt'
# Перевіряємо також копію requirements у Drive до будь-яких змін середовища.
active_requirements = [Requirement(line) for line in requirements.read_text().splitlines()
                       if line.strip() and not line.lstrip().startswith('#')]
if not any(r.name == 'numpy' and (r.marker is None or r.marker.evaluate())
           and r.specifier.contains(numpy_version) for r in active_requirements):
    raise RuntimeError('Оновіть requirements-colab.txt у Drive з локальної папки '
                       'scripts/visibility_localization та повторіть комірку.')

# Зміна бібліотеки, вже імпортованої в ядро, потребує перезапуску.
modules = {'numpy': 'numpy', 'scipy': 'scipy', 'pandas': 'pandas',
           'matplotlib': 'matplotlib', 'Pillow': 'PIL', 'shapely': 'shapely',
           'scikit-image': 'skimage', 'torch': 'torch', 'torchvision': 'torchvision',
           'opencv-python-headless': 'cv2', 'timm': 'timm', 'kornia': 'kornia'}
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
loaded_before = {name: installed_version(name) for name, module in modules.items()
                 if module in sys.modules}
pins = [f'torch=={torch_version}', f'torchvision=={vision_version}']
WORK_ROOT.mkdir(parents=True, exist_ok=True)
constraints = WORK_ROOT / 'torch-constraints.txt'
constraints.write_text('\n'.join(pins) + '\n')
# Без -q: справжня причина помилки pip залишиться у виводі комірки.
subprocess.run([sys.executable, '-m', 'pip', 'install', *pins,
                '--index-url', f'https://download.pytorch.org/whl/{cuda_build}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements),
                '-c', str(constraints)], check=True)
changed = [name for name, version in loaded_before.items()
           if installed_version(name) != version]
restart_message = ('Runtime → Restart session, потім виконайте notebook зверху вниз.')
if changed:
    raise RuntimeError(f'Оновлено вже імпортовані бібліотеки: {changed}. ' + restart_message)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(PACKAGE_DIR/'requirements-ablation-colab.txt'), '-c', str(constraints)], check=True)
import numpy as np, torch, torchvision, cv2
if (np.__version__ != numpy_version
        or torch.__version__.split('+')[0] != torch_version
        or torchvision.__version__.split('+')[0] != vision_version
        or cv2.__version__ != '4.10.0'):
    raise RuntimeError('Версії в ядрі не відповідають встановленим. ' + restart_message)
assert torch.cuda.is_available(), 'GPU CUDA недоступний; перевірте тип runtime'
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                str(PACKAGE_DIR),
                '-o', 'cache_dir=/content/visibility_pytest_cache'], check=True)
from visibility_localization.colab import checkout_upstream
upstream = checkout_upstream(UPSTREAM_ROOT)
from visibility_localization.check_upstream import verify
print('Синтетична перевірка авторських інтерфейсів:', verify(upstream, ablation=True))
output_root = Path(RESULTS_ROOT)/RUN_TAG
output_root.mkdir(parents=True, exist_ok=True)


## 3. Дані, ваги та ідентичність запуску

Якщо сцени вже розпаковані, ZIP повторно не читаються. Інакше використовується наявний механізм отримання вибраних сцен із резервними ZIP у Drive. Вибір кадрів тут явний: без відбору за успішністю. RoMa та DINOv2 завантажуються за офіційними посиланнями, вказаними авторами; CAMP — за їхнім Google Drive. Ваги SP/LG завантажує авторська реалізація.


In [ ]:
from visibility_localization.ablation import (
    prepare_upstream, download_roma_weights, initialize_manifest, author_options,
    crop_controls, search_controls, summarize_controls, seed_all)
from visibility_localization.colab import acquire_dataset, CAMP_URL
from visibility_localization.core import defaults, file_hash
from visibility_localization.campaign import write_json
import pandas as pd

missing_scenes = [scene for scene in SAMPLE_IDS
                  if not list(DATASET_ROOT.rglob(f'{scene}/L??_reference.json'))]
if missing_scenes:
    acquire_dataset(missing_scenes, ARCHIVE_DIR, DATASET_ROOT,
                    output_root/'acquisition.json', archives={s: ARCHIVES[s] for s in missing_scenes})
selection = []
for scene, sample_ids in SAMPLE_IDS.items():
    refs = list(DATASET_ROOT.rglob(f'{scene}/L??_reference.json'))
    if len(refs) != 1:
        raise ValueError(f'Неоднозначна або відсутня сцена: {scene}')
    for sample_id in sample_ids:
        path = refs[0].parent/f'{sample_id}.npz'
        if not path.is_file():
            raise FileNotFoundError(path)
        selection.append(dict(scene_id=scene, sample_id=sample_id,
                              path=str(path), reference_json=str(refs[0])))
display(pd.DataFrame(selection))

weight_hashes = download_roma_weights(upstream)
checkpoint = Path(CAMP_LOCAL_PATH) if CAMP_LOCAL_PATH else upstream/'Retrieval_Models/CAMP/weights/weights_0.9446_for_U1652.pth'
if not checkpoint.is_file():
    import gdown
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    if not gdown.download(CAMP_URL, str(checkpoint), fuzzy=True, quiet=False):
        raise RuntimeError('Не завантажено CAMP; задайте CAMP_LOCAL_PATH')
weight_hashes['CAMP'] = file_hash(checkpoint)
avl, author_runner = prepare_upstream(upstream)
CONFIG = defaults()
CONFIG.update(scenes=list(SAMPLE_IDS), sample_count=len(selection), sigma_xy_m=[20.], trials=1,
              max_image_side=1_000_000)  # native pixels, без додаткового resize до 1600
manifest = initialize_manifest(output_root, upstream, selection, CONFIG, REFERENCE_MODES, weight_hashes)
print('Experiment:', manifest['experiment_id'])
print('Output:', output_root)
print('Очікується', len(selection)*len(REFERENCE_MODES)*7, 'рядків у семи контрольних етапах')


## 4. A/B: авторський RoMa та перевірка адаптера

Ініціалізація RoMa виконується авторською `matching_init`. A виконує оригінальне тіло `process_one_sample` зі спостерігачами входів/виходів. B використовує production `match_candidate` з тим самим авторським crop і правильною зворотною трансформацією повернутої ділянки. B_replay виключає вплив повторного нейромережевого inference; B_live перевіряє також повторний inference з тим самим seed.

Допуски: відповідності/входи PnP ≤0,001 у відповідних одиницях; різниця 3D-пози ≤0,01 м. Порівнюються координати пози, а не лише модуль похибки. Два невдалі PnP не підтверджують правильність обчислення пози.


У controls-v2 повернення відповідностей crop до карти зберігає авторські етапи float32-округлення перед індексацією DSM. Згортання цих етапів в одну афінну матрицю могло змінювати координати й вибрану комірку DSM. Допуски A/B не збільшено. Максимальні розбіжності XY/Z входів PnP записуються окремо.


In [ ]:
seed_all(CONFIG['seed'])
roma_options = author_options(author_runner, upstream, DATASET_ROOT, output_root/'author', REFERENCE_MODES[0])
ROMA_METHODS = avl.matching_init({'matching_method': 'Roma'}, roma_options, {'DEVICE':'cuda'})
roma_results, parity = crop_controls(upstream, DATASET_ROOT, selection, CONFIG, REFERENCE_MODES,
                                    output_root, ROMA_METHODS, roma=True)
display(roma_results)
display(parity)


In [ ]:
if parity.empty or not parity.passed.all():
    failed = parity.loc[~parity.passed].copy() if not parity.empty else parity
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
        display(failed)
    raise RuntimeError('A/B не збігаються: вище показано конкретні розбіжності. '
                       'Перегляньте max_world_xy_delta_m, max_world_z_delta_m, pose_delta_m '
                       'та records/*/correspondences.npz. C/D не виконуються.')
if not parity.pose_verified.any():
    raise RuntimeError('A/B не дали жодної порівнюваної пози. Перевірка адаптера неповна; C/D не виконуються.')
print('A/B: усі перевірки збіглися; позу перевірено в', int(parity.pose_verified.sum()), 'парах.')
import gc
del ROMA_METHODS
gc.collect()
torch.cuda.empty_cache()


## 5. C: змінюємо тільки matcher, потім окремо поворот crop

`C_SP_LG` мусить отримати ті самі фактичні пікселі matcher, що A; хеші перевіряються автоматично. Ваги SP/LG будуть тими самими для C та D. `C_SP_LG_p` окремо показує вплив вимкнення yaw-повороту. Пороги якості проєкту збережено; вони не впливають на raw matches, а лише на прийняття пози.


In [ ]:
from visibility_localization.backend import AnyVisLocBackend
backend = AnyVisLocBackend(upstream, CONFIG, checkpoint)
write_json(output_root/'model_identity.json', backend.identity)
sp_results, _ = crop_controls(upstream, DATASET_ROOT, selection, CONFIG, REFERENCE_MODES,
                              output_root, backend.methods, roma=False)
display(sp_results)


## 6. D: пошук карти, потім шум prior

У D0/D1 однакові SP/LG, native RGB/K/dist, радіус 3σXY, карта та параметри CAMP. D0 використовує точну позу як контроль. D1 змінює лише значення prior за фіксованим шумом. Перехід C_p → D0 змінює підготовку ділянок на пошук і геометрію проєкту; D0 → D1 ізолює шум prior. Еталон не входить у `localize`; оцінка похибки виконується після нього.


In [ ]:
search_results = search_controls(backend, upstream, DATASET_ROOT, selection, CONFIG, REFERENCE_MODES, output_root)
display(search_results)


## 7. Порівняння й збереження висновків

Основна спільна метрика — частка всіх спроб із валідною PnP-позицією та XY-похибкою ≤10 м. Авторське `pnp_success` не дорівнює точній локалізації. `accepted` проєкту зберігається окремо; середні/медіани похибок не підміняють частку успіхів і не включають відсутні пози як нуль.

Порядок діагностики: A/B mismatch → адаптер; A добре, C гірше → matcher; C краще C_p → поворот; C_p краще D0 → пошук/геометрія ділянок; D0 краще D1 → шум prior. Якщо A теж поганий на satellite, порівняйте aerial на тих самих кадрах. Висновки лише для цих трьох кадрів, без узагальнення на весь датасет.


In [ ]:
all_results, summary = summarize_controls(output_root)
expected = len(selection)*len(REFERENCE_MODES)*7
if len(all_results) != expected:
    raise RuntimeError(f'Неповний запуск: {len(all_results)}/{expected}. Не порівнюйте неповні етапи.')
display(summary)
paired_errors = all_results.pivot(index=['reference_mode','scene_id','sample_id'], columns='stage', values='error_xy_m')
paired_errors.to_csv(output_root/'paired_errors.csv')
display(paired_errors)

import matplotlib.pyplot as plt
figure_dir = output_root/'figures'
figure_dir.mkdir(exist_ok=True)
order = ['A_author_Roma','B_adapter_replay','B_adapter_live','C_SP_LG','C_SP_LG_p',
         'D0_search_exact_pose','D1_search_noisy_pose']
for mode in REFERENCE_MODES:
    fig, axes = plt.subplots(1, 2, figsize=(15,5))
    data = all_results.loc[all_results.reference_mode.eq(mode)]
    for (scene,sample_id), group in data.groupby(['scene_id','sample_id']):
        values = group.set_index('stage').reindex(order)
        axes[0].plot(range(len(order)), values.error_xy_m, marker='o', label=f'{scene}/{sample_id}')
        axes[1].plot(range(len(order)), values.inliers, marker='o', label=f'{scene}/{sample_id}')
    axes[0].axhline(10, color='red', linestyle='--', label='10 м')
    axes[0].set(ylabel='XY-похибка всіх наявних PnP-позицій, м', yscale='symlog')
    axes[1].set(ylabel='PnP inliers')
    for ax in axes:
        ax.set_xticks(range(len(order)), order, rotation=60, ha='right')
        ax.grid(alpha=.25); ax.legend(fontsize=8)
    fig.suptitle(mode+'; ті самі кадри, clear, native resolution')
    fig.tight_layout()
    fig.savefig(figure_dir/f'{mode}-controls.png', dpi=150)
    plt.show()
print('Результати та артефакти:', output_root)
print('Постійне збереження у Drive:', SAVE_RESULTS_TO_DRIVE)
